In [5]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import quote
from datetime import datetime, timedelta

# Important notes
You must manually go through the top 300 news articles, check their keywords, search it up on DCInside, and then decide if it is a good article to pursue on. The detailed challenges on this are mentioned on the paper.

After selecting the keyword, you should set the `OUTPUT_FILE`'s number to match it's topic number.

In [6]:
# Set your exact search query here!!!
COMPOUND_KEYWORD = "유빈" 

calculated_start_dt = datetime.strptime("2025-09-01-00-00", "%Y-%m-%d-%H-%M")
calculated_end_dt = datetime.strptime("2026-04-30-23-59", "%Y-%m-%d-%H-%M")
START_DATE_STR = calculated_start_dt.strftime("%Y-%m-%d-%H-%M")
END_DATE_STR = calculated_end_dt.strftime("%Y-%m-%d-%H-%M")

safe_filename_keyword = COMPOUND_KEYWORD.replace(" ", "_")
OUTPUT_FILE = f"../data/3_{safe_filename_keyword}_list.csv"

print(f"Target Query: '{COMPOUND_KEYWORD}'")
print(f"Time Window: {START_DATE_STR} to {END_DATE_STR}")
print(f"Output File: {OUTPUT_FILE}")

Target Query: '유빈'
Time Window: 2025-09-01-00-00 to 2026-04-30-23-59
Output File: ../data/3_유빈_list.csv


In [7]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_custom_encoded_keyword(keyword):
    """
    Converts '김종국 결혼식' -> '.EA.B9.80.EC.A2.85.EA.B5.AD.20.EA.B2.B0.ED.98.BC.EC.8B.9D'
    Standard URL encode uses %, DC Inside uses . for paths.
    """
    encoded = quote(keyword.encode('utf-8')).replace('%', '.')
    return encoded

def parse_date_arg(date_str):
    """Parses input string 'YYYY-MM-DD-HH-MM' to datetime object."""
    return datetime.strptime(date_str, "%Y-%m-%d-%H-%M")

def parse_article_date(date_text):
    """Parses article date '2025.12.28 10:41' to datetime object."""
    return datetime.strptime(date_text, "%Y.%m.%d %H:%M")

In [8]:
def scrape_dcinside():
    start_dt = parse_date_arg(START_DATE_STR)
    end_dt = parse_date_arg(END_DATE_STR)
    
    encoded_keyword = get_custom_encoded_keyword(COMPOUND_KEYWORD)
    
    unique_posts = {}
    
    page = 1
    stop_scraping = False
    
    while not stop_scraping:
        url = f"https://search.dcinside.com/post/p/{page}/sort/latest/q/{encoded_keyword}"
        print(f"Scanning Page {page}: {url}")
        
        try:
            response = requests.get(url, headers=HEADERS)
            if response.status_code != 200:
                print(f"Error: Status code {response.status_code}")
                break
            
            soup = BeautifulSoup(response.text, 'html.parser')
            
            result_list = soup.select('ul.sch_result_list > li')
            
            if not result_list:
                print("No more results found.")
                break
            
            for li in result_list:
                date_element = li.select_one('.link_dsc_txt.dsc_sub .date_time')
                if not date_element:
                    continue
                    
                article_date_str = date_element.get_text(strip=True)
                try:
                    article_dt = parse_article_date(article_date_str)
                except ValueError:
                    print(f"Skipping date parse error: {article_date_str}")
                    continue

                if article_dt > end_dt:
                    continue
                    
                if article_dt < start_dt:
                    print(f"Reached date {article_dt}, which is older than start time. Stopping.")
                    stop_scraping = True
                    break
                
                link_tag = li.select_one('a.tit_txt')
                if link_tag:
                    title = link_tag.get_text(strip=True)
                    article_url = link_tag['href']

                    content_tag = li.select_one('p.link_dsc_txt')
                    content = content_tag.get_text(separator=' ', strip=True) if content_tag else ""
                    unique_posts[title] = [COMPOUND_KEYWORD, article_date_str, title, article_url, content]
        
        except Exception as e:
            print(f"An error occurred: {e}")
            break
        
        page += 1

        if page > 120:
            print("Reached the 120-page limit.")
            break
        
        time.sleep(0.5)

    # After scraping finishes, save the deduplicated results to the CSV
    with open(OUTPUT_FILE, mode='w', newline='', encoding='utf-8-sig') as f:
        writer = csv.writer(f)
        writer.writerow(['keyword', 'time', 'title', 'URL', 'content'])

        # Sort the posts by date(newest first)
        sorted_posts = sorted(unique_posts.values(), key=lambda x: x[1], reverse=True)
        writer.writerows(sorted_posts)
        
    print(f"{len(sorted_posts)} unique posts saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    scrape_dcinside()

Scanning Page 1: https://search.dcinside.com/post/p/1/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 2: https://search.dcinside.com/post/p/2/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 3: https://search.dcinside.com/post/p/3/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 4: https://search.dcinside.com/post/p/4/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 5: https://search.dcinside.com/post/p/5/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 6: https://search.dcinside.com/post/p/6/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 7: https://search.dcinside.com/post/p/7/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 8: https://search.dcinside.com/post/p/8/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 9: https://search.dcinside.com/post/p/9/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 10: https://search.dcinside.com/post/p/10/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 11: https://search.dcinside.com/post/p/11/sort/latest/q/.EC.9C.A0.EB.B9.88
Scanning Page 12: https://search.dcinsi